## Setup and Imports

# CODE-15 ECG Age Prediction with Deep Learning

This notebook implements an advanced ECG-based age prediction system using the CODE-15 dataset with saliency guided learning. **(This notebook contains the training script only. The evaluation script is present in the folder as a python script file titled "CNN-BiLSTM (saliency guided evaluation script"))**:

- **Multi-task learning**: Age regression + bin classification for better calibration
- **Saliency-guided attention**: Focus on ECG regions most relevant for age prediction  
- **5-year bin balancing**: Ensures fair representation across all age groups (16-85 years)
- **Advanced preprocessing**: Optimal bandpass filtering and downsampling pipeline
- **Ensemble methods**: CNN-BiLSTM + XGBoost for improved accuracy
- **Comprehensive monitoring**: Per-decile analysis and artifact-aware evaluation

**Dataset**: CODE-15 (This is the held-out set of ECGs that the CODE ecg age model folks tested their model on)
**Architecture**: CNN → BiLSTM → Multi-task heads → XGBoost ensemble

In [ ]:
# Import all dependencies - organized by category for clarity
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for server compatibility
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import butter, sosfiltfilt, resample_poly, iirnotch, filtfilt, welch
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
import xgboost as xgb
from tqdm import tqdm
import warnings
import h5py
import pickle
from collections import defaultdict

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.experimental.list_physical_devices('GPU')) > 0}")
print("All dependencies loaded successfully")

## Configuration and Global Setup

Instead of treating each age individually, we group them into 15 bins (16-20, 20-25, ..., 85-90).

In [ ]:
# Global configuration - modify these settings as needed
FORCE_CPU = False  # Set True to force CPU-only training (useful for debugging)

if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Define 5-year age bins globally - this is crucial for balanced training
# Using right-open intervals: [16,20), [20,25), ..., [85,90)
BIN_EDGES = np.arange(16, 91, 5)
NUM_BINS = len(BIN_EDGES) - 1  # 15 bins total (indices 0-14)

print(f"Age bin configuration:")
print(f"  Number of bins: {NUM_BINS}")
print(f"  Bin edges: {BIN_EDGES}")
print(f"  Age range: {BIN_EDGES[0]} to {BIN_EDGES[-1]-1} years")

# Set seeds for reproducibility across all libraries
np.random.seed(42)
tf.random.set_seed(42)

print("Configuration completed successfully")

## Core Utility Functions

The age_to_bin_index function is used throughout the pipeline to convert continuous ages into discrete bins for balanced sampling and classification.

In [ ]:
def age_to_bin_index(age):
    """
    Map numeric age to bin index (0 to NUM_BINS-1).
    Uses right-open intervals: [16,20), [20,25), etc.
    
    This is a critical function - it ensures consistent age binning
    throughout the entire pipeline for balanced training.
    """
    age = np.asarray(age, dtype=float)
    # Clamp ages to valid range [16, 90)
    age = np.clip(age, BIN_EDGES[0], BIN_EDGES[-1] - 1e-6)
    # np.digitize returns 1-based indices, convert to 0-based
    idx = np.digitize(age, BIN_EDGES) - 1
    return np.clip(idx, 0, NUM_BINS - 1).astype(int)

def compute_psnr(y_true, y_pred, data_range=None):
    """
    Peak Signal-to-Noise Ratio - borrowed from image processing.
    Provides additional signal quality metric for age predictions.
    Higher PSNR = better prediction quality.
    """
    if data_range is None:
        data_range = np.max(y_true) - np.min(y_true)
    
    mse = np.mean((y_true - y_pred) ** 2)
    if mse == 0:
        return float('inf')
    
    psnr = 20 * np.log10(data_range / np.sqrt(mse))
    return psnr

# Quick test of the age mapping function
print("Testing age-to-bin mapping:")
test_ages = [16, 19.9, 20, 25, 34.9, 50, 65, 85.0, 89.9]
for age in test_ages:
    bin_idx = age_to_bin_index(age)
    print(f"  Age {age:4.1f} → Bin {bin_idx} (range: {BIN_EDGES[bin_idx]:.0f}-{BIN_EDGES[bin_idx+1]:.0f})")

print("Utility functions loaded successfully")

## GPU Setup and Mixed Precision Training

 Mixed precision uses float16 for most operations while keeping float32 for numerical stability where needed. 

In [ ]:
if not FORCE_CPU:
    print("="*60)
    print("CONFIGURING GPU ACCELERATION")
    print("="*60)

    # Detailed GPU detection and configuration
    print(f"TensorFlow version: {tf.__version__}")

    try:
        cuda_available = tf.test.is_built_with_cuda()
        print(f"TensorFlow built with CUDA: {cuda_available}")
    except:
        cuda_available = False
        print("TensorFlow built with CUDA: Unknown")

    # Detect GPUs using both methods for compatibility
    gpus_experimental = tf.config.experimental.list_physical_devices('GPU')
    gpus_list = tf.config.list_physical_devices('GPU')

    print(f"GPUs detected (experimental): {len(gpus_experimental)}")
    print(f"GPUs detected (list_physical): {len(gpus_list)}")

    # Test GPU functionality with a simple operation
    gpu_available = False
    try:
        with tf.device('/GPU:0'):
            test_tensor = tf.constant([1.0, 2.0, 3.0])
            gpu_test_result = tf.reduce_sum(test_tensor)
        print("GPU test operation: SUCCESS")
        gpu_available = True
    except:
        print("GPU test operation: FAILED")
        gpu_available = False

    if gpus_experimental and gpu_available:
        try:
            # Configure GPU memory growth to prevent OOM errors
            for gpu in gpus_experimental:
                tf.config.experimental.set_memory_growth(gpu, True)

            print(f"Configured {len(gpus_experimental)} GPU(s):")
            for i, gpu in enumerate(gpus_experimental):
                print(f"  GPU {i}: {gpu}")

            # Use the first GPU
            tf.config.experimental.set_visible_devices(gpus_experimental[0], 'GPU')

            # Enable mixed precision for faster training on modern GPUs
            policy = tf.keras.mixed_precision.Policy('mixed_float16')
            tf.keras.mixed_precision.set_global_policy(policy)
            print("Mixed precision enabled (float16) - expect 1.5-2x speedup")

            print("GPU configuration successful!")

        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
            print("Falling back to CPU...")
            gpu_available = False
    else:
        gpu_available = False
        print("No GPU available or GPU test failed")
        print("\nTroubleshooting tips:")
        print("1. Install GPU TensorFlow: pip install tensorflow[and-cuda]")
        print("2. Update NVIDIA drivers")
        print("3. Verify CUDA/cuDNN installation")

    print("="*60)
else:
    print("="*60)
    print("FORCED CPU MODE ENABLED")
    print("="*60)
    gpu_available = False

print(f"Final configuration: {'GPU' if gpu_available else 'CPU'} acceleration")

## ECG Signal Processing Pipeline

I filter at 400Hz before downsampling to avoid aliasing artifacts. Using SOS (Second-Order Sections) filters for numerical stability, and resample_poly for downsampling.

Key pipeline: Raw ECG (400Hz) → Optional Notch → Bandpass (0.5-40Hz) → Downsample → Normalize

In [ ]:
def design_bandpass_sos(low_freq, high_freq, fs, order=5):
    """
    Design SOS-based bandpass filter for stable zero-phase filtering.
    Higher order = stronger baseline drift suppression.
    """
    nyquist = fs / 2.0
    low_norm = low_freq / nyquist
    high_norm = high_freq / nyquist
    
    sos = butter(order, [low_norm, high_norm], btype='bandpass', output='sos')
    return sos

def design_notch_sos(center_freq, fs, q_factor=30.0):
    """
    Design notch filter for power line interference removal (50/60Hz).
    Dynamic Q-factor based on sampling rate for optimal performance.
    """
    nyquist = fs / 2.0
    w0 = center_freq / nyquist
    
    # Safety check - avoid frequencies too close to Nyquist
    if w0 >= 0.95:
        print(f"WARNING: Notch frequency {center_freq}Hz too close to Nyquist ({nyquist}Hz)")
        return None
    
    b, a = iirnotch(w0=w0, Q=q_factor)
    return (b, a)

def preprocess_ecg_optimal(ecg_400, train_stats=None, apply_notch=False, fs=400.0):
    """
    Optimal ECG preprocessing pipeline - the heart of signal quality.
    
    Steps:
    1. Optional notch filtering (50/60Hz power line interference)
    2. SOS bandpass filter (0.5-40Hz, order=5) - removes drift and noise
    3. Downsample 400Hz → 100Hz using resample_poly (anti-aliasing built-in)
    4. Ensure exactly 1000 samples (10 seconds at 100Hz)
    5. Z-score normalization per lead using training statistics
    
    CRITICAL: Filter before downsampling to avoid aliasing!
    """
    ecg_working = ecg_400.copy()
    
    # Step 1: Optional notch filtering for power line interference
    if apply_notch:
        for freq in [50.0, 60.0]:  # European and US power line frequencies
            q_factor = max(20.0, fs / 20.0)  # Adaptive Q-factor
            notch_coeffs = design_notch_sos(freq, fs, q_factor)
            
            if notch_coeffs is not None:
                b, a = notch_coeffs
                for lead in range(ecg_working.shape[1]):
                    ecg_working[:, lead] = filtfilt(b, a, ecg_working[:, lead])
    
    # Step 2: Bandpass filter (0.5-40Hz) - removes baseline drift and high-freq noise
    sos = design_bandpass_sos(low_freq=0.5, high_freq=40.0, fs=fs, order=5)
    
    ecg_filtered = np.zeros_like(ecg_working)
    for lead in range(ecg_working.shape[1]):
        ecg_filtered[:, lead] = sosfiltfilt(sos, ecg_working[:, lead])
    
    # Step 3: Efficient downsampling using resample_poly (400Hz → 100Hz)
    downsample_factor = int(fs / 100)
    ecg_100 = resample_poly(ecg_filtered, up=1, down=downsample_factor, axis=0)
    
    # Step 4: Ensure exactly 1000 samples (10 seconds at 100Hz)
    if ecg_100.shape[0] != 1000:
        if ecg_100.shape[0] > 1000:
            ecg_100 = ecg_100[:1000]
        else:
            # Pad with edge values if too short
            pad_needed = 1000 - ecg_100.shape[0]
            ecg_100 = np.pad(ecg_100, ((0, pad_needed), (0, 0)), mode='edge')
    
    # Step 5: Z-score normalization per lead using training statistics
    if train_stats is not None:
        mean = train_stats['mean']
        std = train_stats['std']
        ecg_normalized = (ecg_100 - mean) / std
    else:
        ecg_normalized = ecg_100
    
    return ecg_normalized.astype(np.float32)

# Legacy wrapper for compatibility
def preprocess_trace(ecg_400, train_stats):
    """Legacy wrapper - use preprocess_ecg_optimal instead"""
    return preprocess_ecg_optimal(ecg_400, train_stats)

print("Signal processing functions loaded successfully")
print("Pipeline: Raw → [Notch] → Bandpass → Downsample → Normalize")

## Saliency-Guided Attention Mechanisms

So this is the change from the base CNN BiLSTM script. Instead of random dropout, we use "informed dropout" based on gradient magnitudes - keeping important regions and masking less important ones during training. I believe this might be the reason why this script shaved 0.3 years off the base script's MAE. (6.9 vs 7.2)

In [ ]:
def apply_saliency_masking(batch_ecgs, saliency_mask, keep_scale=1.0, drop_scale=0.0,
                           noise_std=0.02, flip_prob=0.25, dilate_radius=0):
    """
    Apply saliency-guided augmentation for attention guidance.
    
    This is like intelligent dropout - instead of randomly masking parts of the signal,
    we mask based on gradient saliency. This forces the model to focus on the most
    informative parts of the ECG while being robust to less important regions.
    
    Args:
        batch_ecgs: Input ECG batch [B, 1000, 12]
        saliency_mask: Binary mask [1000, 12] where 1=important, 0=less important
        keep_scale: Multiplier for salient regions (usually 1.0)
        drop_scale: Multiplier for non-salient regions (usually 0.0 to zero out)
        noise_std: Add light noise to dropped regions
        flip_prob: Sometimes invert the mask to prevent overfitting
        dilate_radius: Smooth mask edges (in samples)
    """
    B, T, C = batch_ecgs.shape
    mask = saliency_mask.astype(np.float32)
    
    # Optional dilation to soften mask edges
    if dilate_radius > 0:
        k = 2 * dilate_radius + 1
        kernel = np.ones((k,), dtype=np.float32) / k
        m = np.copy(mask)
        for ch in range(C):
            # Simple 1D convolution for smoothing
            padded = np.pad(m[:, ch], (dilate_radius, dilate_radius), mode='edge')
            m[:, ch] = np.convolve(padded, kernel, mode='valid')
        mask = (m > 0.5).astype(np.float32)
    
    # Randomly flip mask to prevent overfitting to specific patterns
    if np.random.rand() < flip_prob:
        mask = 1.0 - mask
    
    # Apply masking to entire batch
    mask_b = np.broadcast_to(mask[None, ...], (B, T, C)).astype(np.float32)
    kept = batch_ecgs * (keep_scale * mask_b)
    dropped = batch_ecgs * (drop_scale * (1.0 - mask_b))
    
    # Add light noise to non-salient regions for regularization
    if noise_std and noise_std > 0:
        noise = np.random.normal(loc=0.0, scale=noise_std, size=batch_ecgs.shape).astype(np.float32)
        dropped = dropped + noise * (1.0 - mask_b)
    
    return kept + dropped

def compute_saliency_mask(model, X_sample, percentile_threshold=25):
    """
    Compute gradient-based saliency mask for attention guidance.
    
    Process:
    1. Forward pass on sample batch
    2. Compute gradients of age prediction w.r.t. input ECG
    3. Take absolute value and average across samples  
    4. Create binary mask using percentile threshold
    
    Args:
        model: Trained model to analyze
        X_sample: ECG samples to compute gradients on
        percentile_threshold: Keep regions above this percentile (25 = top 75%)
    
    Returns:
        mask: Binary mask [1000, 12] where 1=salient, 0=non-salient
        avg_saliency: Raw saliency values before thresholding
    """
    batch_size = min(32, len(X_sample))
    indices = np.random.choice(len(X_sample), batch_size, replace=False)
    
    all_saliency = []
    
    for idx in indices:
        x = X_sample[idx:idx+1]
        x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
        
        with tf.GradientTape() as tape:
            tape.watch(x_tensor)
            outs = model(x_tensor)
            # Use age output for gradient computation
            y_pred = outs[0] if isinstance(outs, (list, tuple)) else outs
        
        grads = tape.gradient(y_pred, x_tensor)
        if grads is not None:
            sal = tf.abs(grads).numpy()[0]  # Shape: [1000, 12]
            all_saliency.append(sal)
    
    if len(all_saliency) == 0:
        print("WARNING: No gradients computed, returning uniform mask")
        return np.ones((1000, 12), dtype=np.float32), np.ones((1000, 12), dtype=np.float32)
    
    # Average saliency across samples
    avg_saliency = np.mean(all_saliency, axis=0)
    
    # Create binary mask based on percentile threshold
    threshold = np.percentile(avg_saliency, percentile_threshold)
    mask = (avg_saliency > threshold).astype(np.float32)
    
    print(f"Saliency mask computed: shape {mask.shape}, {np.mean(mask):.2f} fraction kept")
    
    return mask, avg_saliency

print("Saliency-based attention functions loaded")
print("Features: Gradient-based masking, noise injection, mask flipping")

## Training Monitoring Callbacks

 R² monitor, per-decile monitoring to track consistent performance across age ranges, and batch-level logging to help catch training problems early.

In [ ]:
class R2MonitorCallback(callbacks.Callback):
    """
    Monitor R² per epoch to detect regression-to-mean problems.
    
    This is crucial for age prediction - if R² starts declining significantly,
    it often means the model is overfitting or converging to mean prediction.
    """
    
    def __init__(self, val_dataset, steps_per_epoch, log_frequency=1, max_eval_batches=50):
        super().__init__()
        self.val_dataset = val_dataset
        self.steps_per_epoch = steps_per_epoch
        self.log_frequency = log_frequency
        self.max_eval_batches = max_eval_batches
        self.r2_history = []
        
    def on_epoch_end(self, epoch, logs=None):
        if epoch % self.log_frequency == 0:
            all_preds = []
            all_labels = []
            
            for i, (X_batch, y_batch_dict, w_batch_dict) in enumerate(self.val_dataset):
                if i >= min(self.max_eval_batches, self.steps_per_epoch):
                    break
                    
                model_outputs = self.model(X_batch, training=False)
                age_preds = model_outputs[0] if isinstance(model_outputs, list) else model_outputs
                
                all_preds.extend(age_preds.numpy().flatten())
                all_labels.extend(y_batch_dict['age_output'].numpy())
            
            if len(all_labels) > 0:
                y_true = np.array(all_labels)
                y_pred = np.array(all_preds)
                
                r2 = r2_score(y_true, y_pred)
                mae = mean_absolute_error(y_true, y_pred)
                
                self.r2_history.append(r2)
                
                print(f"Epoch {epoch + 1:3d} - Validation R²: {r2:.4f}, MAE: {mae:.3f} years")
                
                # Detect regression-to-mean trend
                if len(self.r2_history) >= 6:
                    recent_trend = np.mean(self.r2_history[-3:]) - np.mean(self.r2_history[-6:-3])
                    if recent_trend < -0.05:
                        print(f"  ⚠️  WARNING: R² declining trend detected ({recent_trend:.4f})")

class PerBatchMAECallback(callbacks.Callback):
    """
    Log per-batch MAE for the first few epochs to monitor training dynamics.
    Helps identify if the model is learning properly from the very beginning.
    """
    def __init__(self, max_epochs_to_log=3, log_every_n_batches=100):
        super().__init__()
        self.max_epochs_to_log = max_epochs_to_log
        self.log_every_n_batches = log_every_n_batches
        self.batch_count = 0
        self.current_epoch = 0
        self._last_len = 0

    def on_epoch_begin(self, epoch, logs=None):
        self.current_epoch = epoch
        self.batch_count = 0
        self._last_len = 0
        if epoch < self.max_epochs_to_log:
            sys.stdout.write("\n")

    def on_batch_end(self, batch, logs=None):
        if self.current_epoch < self.max_epochs_to_log and (self.batch_count % self.log_every_n_batches == 0):
            age_mae = logs.get('age_output_mae', 0.0)
            age_loss = logs.get('age_output_loss', 0.0)
            total_loss = logs.get('loss', 0.0)
            bin_acc = logs.get('bin_output_accuracy', 0.0)

            line = (f" Batch {self.batch_count + 1}: "
                    f"MAE={age_mae:.4f} | AgeLoss={age_loss:.4f} | "
                    f"BinAcc={bin_acc:.4f} | Loss={total_loss:.4f}")

            # Clean line overwrite for live updates
            pad = max(0, self._last_len - len(line))
            sys.stdout.write("\r" + line + " " * pad)
            sys.stdout.flush()
            self._last_len = len(line)

        self.batch_count += 1

    def on_epoch_end(self, epoch, logs=None):
        if epoch < self.max_epochs_to_log:
            sys.stdout.write("\n")
            sys.stdout.flush()

class PerDecileMonitorCallback(callbacks.Callback):
    """
    Monitor per-decile performance during training.
    
    This ensures the model performs well across ALL age ranges,
    not just the most common ages in the dataset.
    """
    
    def __init__(self, val_dataset, steps_per_epoch, log_frequency=5, max_eval_batches=20):
        super().__init__()
        self.val_dataset = val_dataset
        self.steps_per_epoch = steps_per_epoch
        self.log_frequency = log_frequency
        self.max_eval_batches = max_eval_batches
        
    def on_epoch_end(self, epoch, logs=None):
        if epoch % self.log_frequency == 0:
            print(f"\nEpoch {epoch} - Computing per-decile metrics...")
            
            all_preds = []
            all_labels = []
            
            for i, (X_batch, y_batch_dict, w_batch_dict) in enumerate(self.val_dataset):
                if i >= min(self.max_eval_batches, self.steps_per_epoch):
                    break
                    
                model_outputs = self.model(X_batch, training=False)
                age_preds = model_outputs[0] if isinstance(model_outputs, list) else model_outputs
                
                all_preds.extend(age_preds.numpy().flatten())
                all_labels.extend(y_batch_dict['age_output'].numpy())
            
            if len(all_labels) > 0:
                y_true = np.array(all_labels)
                y_pred = np.array(all_preds)
                
                # Compute per-decile metrics
                deciles = np.percentile(y_true, np.arange(0, 101, 10))
                decile_results = []
                
                for i in range(len(deciles)-1):
                    mask = (y_true >= deciles[i]) & (y_true < deciles[i+1])
                    if np.sum(mask) > 0:
                        decile_mae = mean_absolute_error(y_true[mask], y_pred[mask])
                        decile_r2 = r2_score(y_true[mask], y_pred[mask])
                        decile_results.append(f"D{i+1}: MAE={decile_mae:.2f}, R²={decile_r2:.3f}")
                
                print(f"Per-decile performance: {' | '.join(decile_results[:5])}...")

class SaliencyGuidedLoss(tf.keras.losses.Loss):
    """
    Wrapper for saliency-guided loss (though guidance actually happens via data masking).
    This is more of a placeholder - the real saliency guidance happens in the data generator.
    """
    
    def __init__(self, base_loss, name="saliency_guided_loss"):
        super().__init__(name=name)
        self.base_loss = base_loss
        
    def call(self, y_true, y_pred):
        return self.base_loss(y_true, y_pred)

print("Custom training callbacks loaded successfully")
print("Features: R² monitoring, per-batch logging, decile analysis, saliency loss")

## CNN-BiLSTM Model Architecture

I tried to preserve the time dimension through the LSTM layers, then pool globally. To capture both local ECG patterns (CNN) and long-range temporal dependencies (BiLSTM).

Multi-task learning predicts both continuous age and discrete age bins.

In [ ]:
def build_model(input_shape=(1000, 12), median_age=50.0):
    """
    Build CNN-BiLSTM model with multi-task head.
    
    Architecture philosophy:
    - CNN layers extract local ECG features (P waves, QRS complexes, T waves)
    - BiLSTM captures temporal dependencies and heart rate variability patterns
    - Global pooling summarizes the entire 10-second ECG
    - Multi-task heads provide age + bin predictions for better calibration
    
    Args:
        input_shape: (time_steps, leads) = (1000, 12) for 10s at 100Hz
        median_age: Used to initialize age output bias for faster convergence
    """
    inputs = layers.Input(shape=input_shape, name='ecg_input')
    x = inputs
    
    # Conv Block 1: Local feature extraction
    # Kernel size 7 captures typical QRS duration (~80-120ms at 100Hz)
    x = layers.Conv1D(32, 7, padding='same', name='conv1')(x)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.ReLU(name='relu1')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool1')(x)  # 1000 → 500 samples
    
    # Conv Block 2: Higher-level pattern recognition
    # Kernel size 5 for broader temporal patterns
    x = layers.Conv1D(64, 5, padding='same', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.ReLU(name='relu2')(x)
    x = layers.MaxPooling1D(pool_size=2, name='pool2')(x)  # 500 → 250 samples
    
    # BiLSTM: Temporal sequence modeling - THE CORE OF THE ARCHITECTURE
    # Bidirectional allows the model to see both past and future context
    # This is crucial for ECG analysis where wave morphology depends on context
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, name='lstm'), 
        name='bi_lstm'
    )(x)
    
    # Global pooling: Collapse time dimension AFTER temporal modeling
    # This preserves all temporal information through the LSTM
    shared = layers.GlobalAveragePooling1D(name='global_pool')(x)
    
    # Multi-task heads for improved training and calibration
    
    # Age regression head with smart bias initialization
    age_output = layers.Dense(
        1, 
        activation='linear', 
        name='age_output',
        bias_initializer=tf.keras.initializers.Constant(median_age)  # Start near median
    )(shared)
    
    # Bin classification head for additional supervision
    bin_output = layers.Dense(
        NUM_BINS, 
        activation='softmax', 
        name='bin_output'
    )(shared)
    
    # Ensure float32 output for mixed precision compatibility
    if tf.keras.mixed_precision.global_policy().name == 'mixed_float16':
        age_output = layers.Activation('linear', dtype='float32', name='age_float32')(age_output)
        bin_output = layers.Activation('softmax', dtype='float32', name='bin_float32')(bin_output)

    model = models.Model(inputs=inputs, outputs=[age_output, bin_output], name='ECG_Age_Predictor')
    return model

# Test the architecture
print("Testing model architecture...")
test_model = build_model()
print(f"✓ Model created with {test_model.count_params():,} parameters")
print("\nModel summary:")
test_model.summary()

# Clean up test model to save memory
del test_model
print("Architecture validation completed")

## CODE-15 Dataset Loader with Traveling Fixes
 Key features include patient-level splitting (prevents data leakage), HDF5 file management, and train-only normalization statistics computation.

In [ ]:
class CODE15DatasetFinal:
    """
    CODE-15 dataset loader implementing all traveling fixes and optimizations.
    
    Key improvements:
    - HDF5 efficient loading
    - Patient-level stratified splits (no data leakage)
    - Train-only normalization statistics
    - 5-year bin equalization weights
    - Comprehensive data validation
    """

    def __init__(self, data_dir, metadata_path, target_sampling_rate=100):
        self.data_dir = data_dir
        self.metadata_path = metadata_path
        self.original_sampling_rate = 400  # CODE-15 native rate
        self.target_sampling_rate = target_sampling_rate

        print("Initializing CODE-15 dataset loader...")
        print(f"Data directory: {data_dir}")
        print(f"Metadata file: {metadata_path}")

        # Load and validate metadata CSV
        self.metadata = pd.read_csv(metadata_path)
        print(f"Available metadata columns: {list(self.metadata.columns)}")

        # Validate required columns exist
        required_cols = ['exam_id', 'age', 'patient_id', 'trace_file']
        missing_cols = [col for col in required_cols if col not in self.metadata.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns in metadata: {missing_cols}")

        # Data cleaning and filtering
        print(f"Original dataset size: {len(self.metadata):,} records")
        
        # Remove invalid entries
        self.metadata = self.metadata.dropna(subset=['age', 'patient_id'])
        
        # Filter to target age range (16-85 years)
        age_mask = (self.metadata['age'] >= 16) & (self.metadata['age'] <= 85)
        self.metadata = self.metadata[age_mask]
        
        print(f"After filtering (age 16-85, no NaN): {len(self.metadata):,} records")
        print(f"Age range: {self.metadata['age'].min():.1f} - {self.metadata['age'].max():.1f} years")
        print(f"Unique patients: {self.metadata['patient_id'].nunique():,}")

        # Discover available HDF5 files
        self.hdf5_files = self._discover_hdf5_files()
        print(f"Found {len(self.hdf5_files)} HDF5 data files")

        # Initialize placeholders (computed during training)
        self.train_stats = None
        self.age_weight_by_exam_id = {}
        self.train_median_age = None

    def _discover_hdf5_files(self):
        """Discover and validate HDF5 files in the data directory"""
        hdf5_files = {}
        
        if not os.path.exists(self.data_dir):
            raise FileNotFoundError(f"Data directory not found: {self.data_dir}")
        
        for file in os.listdir(self.data_dir):
            if file.endswith('.hdf5') and file.startswith('exams_part'):
                file_path = os.path.join(self.data_dir, file)
                hdf5_files[file] = file_path
                print(f"  Found HDF5 file: {file}")
        
        if not hdf5_files:
            raise FileNotFoundError("No HDF5 files found matching pattern 'exams_part*.hdf5'")
            
        return hdf5_files

    def _load_ecg_from_hdf5(self, trace_file, exam_id):
        """
        Load ECG data from HDF5 file.
        
        CODE-15 stores ECG data in HDF5 format for efficiency:
        - tracings: ECG data arrays [N, 4096, 12]  
        - exam_id: Corresponding exam IDs [N]
        """
        if trace_file not in self.hdf5_files:
            return None

        try:
            with h5py.File(self.hdf5_files[trace_file], 'r') as f:
                # Load exam IDs and find the target exam
                exam_ids = f['exam_id'][:]
                tracings = f['tracings']

                # Find index for this specific exam_id
                exam_idx = np.where(exam_ids == exam_id)[0]
                if len(exam_idx) == 0:
                    return None

                # Load ECG data: shape should be (4096, 12) = 10.24s at 400Hz, 12 leads
                ecg_data = tracings[exam_idx[0]]
                
                # Validate data shape
                if ecg_data.shape != (4096, 12):
                    print(f"Warning: Unexpected ECG shape {ecg_data.shape} for exam {exam_id}")
                
                return ecg_data.astype(np.float32)

        except Exception as e:
            print(f"Error loading {trace_file}, exam {exam_id}: {str(e)}")
            return None

print("Dataset loading class defined successfully")
print("Features: HDF5 loading, data validation, patient-level organization")

## Advanced Data Generation with Age Bin Balancing

This is the most complex part of the data pipeline. Instead of random sampling, we ensure each batch contains representative samples from all age bins. To prevent the model from only learning about common ages (40-60) and forces it to understand the full age spectrum.

The generator also applies saliency-guided masking during training for attention guidance.

In [ ]:
def compute_train_normalization_stats(self, train_df, sample_size=1000):
    """
    Compute normalization statistics from TRAIN DATA ONLY.
    
    This is absolutely critical - using test/val data here would be data leakage!
    We sample from training data, preprocess without normalization, then compute
    per-lead mean/std statistics.
    """
    print(f"Computing normalization stats from {sample_size} TRAIN samples...")

    all_ecg_data = []
    collected = 0

    # Randomly sample from training data
    sampled_train = train_df.sample(min(sample_size, len(train_df)), random_state=42)

    for idx, row in sampled_train.iterrows():
        try:
            trace_file = row['trace_file']
            exam_id = row['exam_id']

            if trace_file in self.hdf5_files:
                ecg_data = self._load_ecg_from_hdf5(trace_file, exam_id)
                if ecg_data is not None:
                    # Preprocess WITHOUT normalization to get clean signals
                    ecg_processed = preprocess_ecg_optimal(ecg_data, train_stats=None)
                    all_ecg_data.append(ecg_processed)
                    collected += 1

            if collected >= sample_size:
                break

        except Exception as e:
            continue

    if len(all_ecg_data) == 0:
        print("WARNING: No ECG data loaded for stats computation!")
        return {'mean': np.zeros(12), 'std': np.ones(12)}

    # Compute per-lead statistics across all timepoints
    all_data = np.vstack(all_ecg_data)  # Shape: (N*1000, 12)

    lead_means = np.mean(all_data, axis=0)  # Shape: (12,)
    lead_stds = np.std(all_data, axis=0)    # Shape: (12,)
    lead_stds = np.maximum(lead_stds, 1e-8)  # Prevent division by zero

    self.train_stats = {'mean': lead_means, 'std': lead_stds}

    print(f"✓ Computed stats from {collected} TRAIN samples")
    print(f"  Mean range: [{lead_means.min():.4f}, {lead_means.max():.4f}]")
    print(f"  Std range: [{lead_stds.min():.4f}, {lead_stds.max():.4f}]")

    return self.train_stats

def compute_age_weights_from_train(self, train_df):
    """
    Compute 5-year bin equalization weights from TRAIN DATA ONLY.
    
    Instead of weighting individual ages (unstable), we group ages into 5-year bins
    and compute inverse frequency weights. This gives higher weights to rare age groups.
    """
    print("Computing 5-year bin equalization weights from TRAIN data...")
    
    train_ages = train_df['age'].values
    
    # Map ages to bin indices using our global function
    train_bins = np.array([age_to_bin_index(a) for a in train_ages])
    bincounts = np.bincount(train_bins, minlength=NUM_BINS)
    
    print(f"Bin counts per 5-year group: {bincounts}")
    print("Bin distributions:")
    for i, count in enumerate(bincounts):
        age_range = f"{BIN_EDGES[i]:.0f}-{BIN_EDGES[i+1]:.0f}"
        print(f"  {age_range} years: {count:,} samples ({100*count/len(train_ages):.1f}%)")

    # Compute inverse frequency weights, normalized to mean=1
    counts = np.maximum(bincounts, 1)  # Avoid division by zero
    w_per_bin = counts.mean() / counts  # Higher weight for rarer bins
    raw_weights = w_per_bin[train_bins]
    
    # Clip weights to reasonable range to prevent extreme values
    age_weights = np.clip(raw_weights, 0.5, 3.0)
    age_weights = age_weights / age_weights.mean()  # Normalize to mean=1

    print(f"Weight statistics: min={age_weights.min():.3f}, mean={age_weights.mean():.3f}, max={age_weights.max():.3f}")

    # Build efficient lookup dictionary by exam_id
    self.age_weight_by_exam_id = {}
    for i, (_, row) in enumerate(train_df.reset_index(drop=True).iterrows()):
        exam_id = row['exam_id']
        weight = age_weights[i]
        self.age_weight_by_exam_id[exam_id] = weight

    print(f"✓ Built weight lookup for {len(self.age_weight_by_exam_id):,} train exam_ids")

    # Store median age for model bias initialization
    self.train_median_age = float(np.median(train_ages))
    print(f"Train median age: {self.train_median_age:.1f} years")

    return self.age_weight_by_exam_id

def prepare_stratified_patient_splits(self, test_size=0.15, val_size=0.05):
## Chunk 10: Data Generator with Bin Balancing

### Markdown Cell:
```markdown
## Advanced Data Generation with Age Bin Balancing

This is the most complex part of the data pipeline. Instead of random sampling, we ensure each batch contains representative samples from all age bins. This prevents the model from only learning about common ages (40-60) and forces it to understand the full age spectrum.

The generator also applies saliency-guided masking during training for attention guidance.

In [ ]:
def compute_train_normalization_stats(self, train_df, sample_size=1000):
    """
    Compute normalization statistics from TRAIN DATA ONLY.
    
    This is absolutely critical - using test/val data here would be data leakage!
    We sample from training data, preprocess without normalization, then compute
    per-lead mean/std statistics.
    """
    print(f"Computing normalization stats from {sample_size} TRAIN samples...")

    all_ecg_data = []
    collected = 0

    # Randomly sample from training data
    sampled_train = train_df.sample(min(sample_size, len(train_df)), random_state=42)

    for idx, row in sampled_train.iterrows():
        try:
            trace_file = row['trace_file']
            exam_id = row['exam_id']

            if trace_file in self.hdf5_files:
                ecg_data = self._load_ecg_from_hdf5(trace_file, exam_id)
                if ecg_data is not None:
                    # Preprocess WITHOUT normalization to get clean signals
                    ecg_processed = preprocess_ecg_optimal(ecg_data, train_stats=None)
                    all_ecg_data.append(ecg_processed)
                    collected += 1

            if collected >= sample_size:
                break

        except Exception as e:
            continue

    if len(all_ecg_data) == 0:
        print("WARNING: No ECG data loaded for stats computation!")
        return {'mean': np.zeros(12), 'std': np.ones(12)}

    # Compute per-lead statistics across all timepoints
    all_data = np.vstack(all_ecg_data)  # Shape: (N*1000, 12)

    lead_means = np.mean(all_data, axis=0)  # Shape: (12,)
    lead_stds = np.std(all_data, axis=0)    # Shape: (12,)
    lead_stds = np.maximum(lead_stds, 1e-8)  # Prevent division by zero

    self.train_stats = {'mean': lead_means, 'std': lead_stds}

    print(f"✓ Computed stats from {collected} TRAIN samples")
    print(f"  Mean range: [{lead_means.min():.4f}, {lead_means.max():.4f}]")
    print(f"  Std range: [{lead_stds.min():.4f}, {lead_stds.max():.4f}]")

    return self.train_stats

def compute_age_weights_from_train(self, train_df):
    """
    Compute 5-year bin equalization weights from TRAIN DATA ONLY.
    
    Instead of weighting individual ages (unstable), we group ages into 5-year bins
    and compute inverse frequency weights. This gives higher weights to rare age groups.
    """
    print("Computing 5-year bin equalization weights from TRAIN data...")
    
    train_ages = train_df['age'].values
    
    # Map ages to bin indices using our global function
    train_bins = np.array([age_to_bin_index(a) for a in train_ages])
    bincounts = np.bincount(train_bins, minlength=NUM_BINS)
    
    print(f"Bin counts per 5-year group: {bincounts}")

    # Compute inverse frequency weights, normalized to mean=1
    counts = np.maximum(bincounts, 1)  # Avoid division by zero
    w_per_bin = counts.mean() / counts  # Higher weight for rarer bins
    raw_weights = w_per_bin[train_bins]
    
    # Clip weights to reasonable range to prevent extreme values
    age_weights = np.clip(raw_weights, 0.5, 3.0)
    age_weights = age_weights / age_weights.mean()  # Normalize to mean=1

    print(f"Weight statistics: min={age_weights.min():.3f}, mean={age_weights.mean():.3f}, max={age_weights.max():.3f}")

    # Build efficient lookup dictionary by exam_id
    self.age_weight_by_exam_id = {}
    for i, (_, row) in enumerate(train_df.reset_index(drop=True).iterrows()):
        exam_id = row['exam_id']
        weight = age_weights[i]
        self.age_weight_by_exam_id[exam_id] = weight

    print(f"✓ Built weight lookup for {len(self.age_weight_by_exam_id):,} train exam_ids")

    # Store median age for model bias initialization
    self.train_median_age = float(np.median(train_ages))
    print(f"Train median age: {self.train_median_age:.1f} years")

    return self.age_weight_by_exam_id

def prepare_stratified_patient_splits(self, test_size=0.15, val_size=0.05):
    """
    Create stratified patient-level splits to prevent data leakage.
    
    CRITICAL: We split by PATIENT, not by ECG record. This ensures that all
    ECGs from the same patient stay in the same split, preventing the model
    from memorizing patient-specific patterns.
    """
    print("Creating stratified patient-level splits...")

    # Get unique patients with their ages
    patient_data = self.metadata.groupby('patient_id').agg({
        'age': 'first'  # All records from same patient should have same age
    }).reset_index()

    print(f"Total unique patients: {len(patient_data):,}")
    
    # Clean patient data
    patient_data = patient_data.dropna(subset=['age'])
    patient_data = patient_data[(patient_data['age'] >= 16) & (patient_data['age'] <= 85)]
    print(f"Valid patients for splitting: {len(patient_data):,}")

    if len(patient_data) == 0:
        raise ValueError("No valid patients found after filtering!")

    # Create age groups for stratification (coarser than our 5-year bins)
    age_min = patient_data['age'].min()
    age_max = patient_data['age'].max()
    bin_start = int(np.floor(age_min / 10) * 10)  # 10-year groups for stratification
    bin_end = int(np.ceil(age_max / 10) * 10) + 10
    bins = list(range(bin_start, bin_end + 1, 10))
    
    age_groups = pd.cut(patient_data['age'], bins=bins, include_lowest=True, right=False)
    patient_data['age_group'] = age_groups

    # Handle NaN age groups
    if patient_data['age_group'].isna().any():
        patient_data['age_group_str'] = patient_data['age_group'].astype(str)
        age_group_labels = patient_data['age_group_str'].values
    else:
        age_group_labels = patient_data['age_group'].values

    print("Age group distribution for stratification:")
    print(pd.Series(age_group_labels).value_counts().sort_index())

    # Ensure all groups have enough samples for splitting
    age_group_counts = pd.Series(age_group_labels).value_counts()
    valid_groups = age_group_counts[age_group_counts >= 2].index
    if len(valid_groups) < len(age_group_counts):
        mask = pd.Series(age_group_labels).isin(valid_groups)
        patient_data = patient_data[mask].reset_index(drop=True)
        age_group_labels = age_group_labels[mask]

    unique_patients = patient_data['patient_id'].values

    # Two-stage stratified splitting
    # Stage 1: train+val vs test
    train_val_patients, test_patients = train_test_split(
        unique_patients,
        test_size=test_size,
        stratify=age_group_labels,
        random_state=42
    )

    # Stage 2: train vs val
    train_val_mask = patient_data['patient_id'].isin(train_val_patients)
    train_val_age_groups = age_group_labels[train_val_mask]

    train_patients, val_patients = train_test_split(
        train_val_patients,
        test_size=val_size/(1-test_size),  # Adjust for nested splitting
        stratify=train_val_age_groups,
        random_state=42
    )

    # Map patients back to ECG records
    train_data = self.metadata[self.metadata['patient_id'].isin(train_patients)]
    val_data = self.metadata[self.metadata['patient_id'].isin(val_patients)]
    test_data = self.metadata[self.metadata['patient_id'].isin(test_patients)]

    print(f"\nFinal splits:")
    print(f"  Train: {len(train_patients):,} patients, {len(train_data):,} ECG records")
    print(f"  Val:   {len(val_patients):,} patients, {len(val_data):,} ECG records")
    print(f"  Test:  {len(test_patients):,} patients, {len(test_data):,} ECG records")

    return train_data, val_data, test_data

# Add these methods to the dataset class
CODE15DatasetFinal.compute_train_normalization_stats = compute_train_normalization_stats
CODE15DatasetFinal.compute_age_weights_from_train = compute_age_weights_from_train
CODE15DatasetFinal.prepare_stratified_patient_splits = prepare_stratified_patient_splits

print("Dataset statistics and splitting methods added successfully")

## Bin-Balanced Data Generator


In [ ]:
def create_weighted_data_generator(self, df, batch_size=32, shuffle=True, is_train=True, saliency_mask=None):
    """
    Create bin-balanced data generator with multi-task outputs and saliency masking.
    
    This is the heart of the training pipeline. Instead of random sampling, we ensure
    each batch contains representative samples from all 15 age bins. This is much more
    complex than standard data generators but essential for good age prediction.
    
    Args:
        df: DataFrame with ECG records
        batch_size: Target batch size
        shuffle: Whether to shuffle data each epoch
        is_train: If True, apply age weights and saliency masking
        saliency_mask: Gradient-based attention mask for training augmentation
    """
    # Pre-filter valid data to avoid runtime failures
    print(f"Pre-filtering {len(df):,} records for data generator...")
    valid_data = []

    for idx, row in df.iterrows():
        try:
            trace_file = row['trace_file']
            exam_id = row['exam_id']

            if trace_file in self.hdf5_files:
                # Quick validation that ECG can be loaded
                ecg_data = self._load_ecg_from_hdf5(trace_file, exam_id)
                if ecg_data is not None:
                    # Get sample weight
                    if is_train and exam_id in self.age_weight_by_exam_id:
                        weight = self.age_weight_by_exam_id[exam_id]
                    else:
                        weight = 1.0
                    
                    valid_data.append({
                        'exam_id': exam_id,
                        'row': row,
                        'weight': weight
                    })
        except:
            continue

    print(f"✓ Found {len(valid_data):,} valid records ({100*len(valid_data)/len(df):.1f}% success rate)")
    
    # Partition data into bins for balanced sampling
    buckets = [[] for _ in range(NUM_BINS)]
    for item in valid_data:
        age = item['row']['age']
        bin_idx = age_to_bin_index(age)
        if 0 <= bin_idx < NUM_BINS:
            buckets[bin_idx].append(item)
    
    # Print bin distribution
    bin_counts = [len(bucket) for bucket in buckets]
    print(f"Bin distribution: {bin_counts}")
    for i, count in enumerate(bin_counts):
        if count > 0:
            age_range = f"{BIN_EDGES[i]:.0f}-{BIN_EDGES[i+1]:.0f}"
            print(f"  Bin {i:2d} ({age_range} years): {count:4d} samples")
    
    def generator():
        """Inner generator function that yields infinite batches"""
        while True:
            # Create fresh bucket copies for each epoch
            bucket_copies = [bucket.copy() for bucket in buckets]
            if shuffle:
                for bucket in bucket_copies:
                    np.random.shuffle(bucket)

            # Generate balanced batches until buckets are exhausted
            while any(len(bucket) > 0 for bucket in bucket_copies):
                batch_ecgs = []
                batch_ages = []
                batch_bins = []
                batch_weights = []

                # Determine samples per bin for this batch
                base_per_bin = max(1, batch_size // NUM_BINS)
                remainder = batch_size - (base_per_bin * NUM_BINS)
                
                # Distribute remainder among first few bins
                for bin_idx in range(NUM_BINS):
                    samples_needed = base_per_bin + (1 if bin_idx < remainder else 0)
                    
                    for _ in range(samples_needed):
                        # Try to get sample from current bin
                        if len(bucket_copies[bin_idx]) > 0:
                            item = bucket_copies[bin_idx].pop(0)
                        else:
                            # Sample with replacement if bin is empty
                            if len(buckets[bin_idx]) > 0:
                                item = np.random.choice(buckets[bin_idx])
                            else:
                                continue  # Skip if bin has no samples at all
                        
                        try:
                            row = item['row']
                            exam_id = item['exam_id']
                            trace_file = row['trace_file']

                            # Load and preprocess ECG
                            ecg_data = self._load_ecg_from_hdf5(trace_file, exam_id)
                            if ecg_data is not None:
                                ecg_processed = preprocess_ecg_optimal(ecg_data, self.train_stats)
                                
                                y_age = row['age']
                                y_bin = age_to_bin_index(y_age)
                                
                                batch_ecgs.append(ecg_processed)
                                batch_ages.append(y_age)
                                batch_bins.append(y_bin)
                                batch_weights.append(item['weight'])
                        except:
                            continue

                # Yield complete batch
                if len(batch_ecgs) == batch_size:
                    batch_ecgs_array = np.array(batch_ecgs)
                    
                    # Apply saliency masking for training data
                    if is_train and saliency_mask is not None:
                        batch_ecgs_array = apply_saliency_masking(batch_ecgs_array, saliency_mask)
                    
                    # Multi-task output format
                    yield (
                        tf.cast(batch_ecgs_array, tf.float32),
                        {
                            'age_output': tf.cast(np.array(batch_ages), tf.float32),
                            'bin_output': tf.cast(np.array(batch_bins), tf.int32)
                        },
                        {
                            'age_output': tf.cast(np.array(batch_weights), tf.float32),
                            'bin_output': tf.ones((len(batch_bins),), dtype=tf.float32)
                        }
                    )

    return generator, len(valid_data)

# Add the generator method to the dataset class
CODE15DatasetFinal.create_weighted_data_generator = create_weighted_data_generator

print("Bin-balanced data generator implemented successfully")
print("Features: Age bin balancing, saliency masking, multi-task outputs, sample weighting")

## Complete Model Class with Ensemble Capability

This class manages the entire model pipeline: CNN-LSTM for deep learning, feature extraction for XGBoost, and ensemble predictions.

In [ ]:
class CNNLSTMModelFinal:
    """
    Complete CNN-LSTM model with multi-task head, Huber loss, and XGBoost ensemble.
    
    This class orchestrates:
    - CNN-LSTM model training with multi-task outputs
    - Feature extraction for XGBoost
    - Ensemble prediction combining both models
    - Saliency mask management for attention guidance
    """

    def __init__(self, input_shape=(1000, 12), median_age=50.0):
        self.input_shape = input_shape
        self.median_age = median_age
        
        # Model components
        self.model = None
        self.xgb_model = None
        self.feature_extractor = None
        self.saliency_mask = None
        
        # Loss configuration
        self.base_huber_loss = tf.keras.losses.Huber(delta=5.0)  # Robust to outliers

    def build_model(self, use_saliency_guidance=True):
        """Build and compile the multi-task CNN-LSTM model"""
        print("Building CNN-LSTM model with multi-task head...")
        
        # Create the model architecture
        self.model = build_model(self.input_shape, self.median_age)
        
        # Create feature extractor (extract features before final dense layers)
        shared_layer = None
        for layer in self.model.layers:
            if isinstance(layer, layers.GlobalAveragePooling1D):
                shared_layer = layer
                break
        
        if shared_layer is not None:
            self.feature_extractor = models.Model(
                inputs=self.model.input, 
                outputs=shared_layer.output
            )
            print("✓ Feature extractor created for XGBoost ensemble")

        # Configure optimizer with gradient clipping
        optimizer = Adam(
            learning_rate=0.001,
            clipnorm=1.0,  # Gradient clipping for stability
            epsilon=1e-4   # Numerical stability
        )

        # Setup loss functions
        if use_saliency_guidance:
            age_loss = SaliencyGuidedLoss(self.base_huber_loss)
            print("✓ Saliency-guided loss wrapper applied")
        else:
            age_loss = self.base_huber_loss

        # Compile with multi-task configuration
        self.model.compile(
            optimizer=optimizer,
            loss={
                'age_output': age_loss, 
                'bin_output': 'sparse_categorical_crossentropy'
            },
            loss_weights={
                'age_output': 1.0,  # Primary task
                'bin_output': 0.2   # Auxiliary task for calibration
            },
            metrics={
                'age_output': ['mae'], 
                'bin_output': ['accuracy']
            },
            run_eagerly=False
        )

        print("✓ Model compiled successfully")
        print(f"  Architecture: Conv1D → BiLSTM → GlobalPool → [Age + Bin] heads")
        print(f"  Loss: Huber (δ=5) for age, SparseCategoricalCE for bins")
        print(f"  Parameters: {self.model.count_params():,}")

        return self.model

    def update_saliency_mask(self, new_mask):
        """Update saliency mask for attention guidance"""
        self.saliency_mask = new_mask
        print(f"Saliency mask updated: {new_mask.shape}, {np.mean(new_mask):.2f} fraction active")

    def evaluate_standalone(self, test_dataset, steps=None):
        """Evaluate CNN-LSTM model before XGBoost ensemble"""
        print("Evaluating CNN-LSTM standalone performance...")
        
        all_preds_age = []
        all_preds_bin = []
        all_labels_age = []
        all_labels_bin = []
        
        step_count = 0
        for X_batch, y_batch_dict, w_batch_dict in test_dataset:
            if steps and step_count >= steps:
                break
                
            model_outputs = self.model.predict(X_batch, verbose=0)
            age_preds = model_outputs[0].flatten()
            bin_preds = np.argmax(model_outputs[1], axis=1)
            
            all_preds_age.extend(age_preds)
            all_preds_bin.extend(bin_preds)
            all_labels_age.extend(y_batch_dict['age_output'].numpy())
            all_labels_bin.extend(y_batch_dict['bin_output'].numpy())
            
            step_count += 1
        
        # Compute comprehensive metrics
        y_true_age = np.array(all_labels_age)
        y_pred_age = np.array(all_preds_age)
        y_true_bin = np.array(all_labels_bin)
        y_pred_bin = np.array(all_preds_bin)
        
        # Age regression metrics
        mae = mean_absolute_error(y_true_age, y_pred_age)
        rmse = np.sqrt(mean_squared_error(y_true_age, y_pred_age))
        r2 = r2_score(y_true_age, y_pred_age)
        pearson_r, _ = pearsonr(y_true_age, y_pred_age)
        psnr = compute_psnr(y_true_age, y_pred_age)
        
        # Bin classification metrics
        bin_accuracy = np.mean(y_true_bin == y_pred_bin)
        
        results = {
            'mae': mae, 'rmse': rmse, 'r2': r2, 'pearson_r': pearson_r,
            'psnr': psnr, 'bin_accuracy': bin_accuracy, 'n_samples': len(y_true_age)
        }
        
        print(f"Standalone Results:")
        print(f"  Age: MAE={mae:.3f}y, RMSE={rmse:.3f}y, R²={r2:.3f}, PSNR={psnr:.1f}dB")
        print(f"  Bin: Accuracy={bin_accuracy:.3f} ({len(y_true_age):,} samples)")
        
        return results

    def extract_features(self, X):
        """Extract deep features for XGBoost ensemble"""
        if self.feature_extractor is None:
            raise ValueError("Feature extractor not available. Build model first.")
        return self.feature_extractor.predict(X, verbose=0)

    def train_xgboost(self, X_train, y_train, X_val, y_val, output_dir='./results'):
        """Train XGBoost using extracted CNN-LSTM features"""
        print("Training XGBoost ensemble component...")

        # Extract features using the trained CNN-LSTM
        print("Extracting features from CNN-LSTM...")
        train_features = self.extract_features(X_train)
        val_features = self.extract_features(X_val)
        
        print(f"Feature shape: {train_features.shape}")

        # XGBoost hyperparameters (tuned for age regression)
        xgb_params = {
            'n_estimators': 300,
            'max_depth': 6,
            'learning_rate': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,      # L1 regularization
            'reg_lambda': 1.0,     # L2 regularization
            'objective': 'reg:squarederror',
            'random_state': 42,
            'n_jobs': -1,
            'verbosity': 0
        }

        # Try GPU acceleration for XGBoost
        gpu_detected = len(tf.config.experimental.list_physical_devices('GPU')) > 0
        if gpu_detected:
            try:
                xgb_params['tree_method'] = 'gpu_hist'
                xgb_params['gpu_id'] = 0
                print("✓ XGBoost GPU acceleration enabled")
            except:
                print("XGBoost GPU not available, using CPU")

        # Train XGBoost
        self.xgb_model = xgb.XGBRegressor(**xgb_params)
        
        self.xgb_model.fit(
            train_features, y_train,
            eval_set=[(val_features, y_val)],
            verbose=False
        )

        # Evaluate XGBoost standalone
        val_pred = self.xgb_model.predict(val_features)
        val_mae = mean_absolute_error(y_val, val_pred)
        val_r2 = r2_score(y_val, val_pred)

        print(f"✓ XGBoost trained: MAE={val_mae:.3f}y, R²={val_r2:.3f}")

        return val_mae, val_r2

    def predict_ensemble(self, X, cnn_weight=0.6, xgb_weight=0.4):
        """Generate ensemble predictions combining CNN-LSTM and XGBoost"""
        # CNN-LSTM predictions (use age output only)
        model_outputs = self.model.predict(X, verbose=0)
        cnn_pred = model_outputs[0].flatten()
        
        # XGBoost predictions on extracted features
        features = self.extract_features(X)
        xgb_pred = self.xgb_model.predict(features)
        
        # Weighted ensemble
        ensemble_pred = cnn_weight * cnn_pred + xgb_weight * xgb_pred
        
        # Clip to valid age range (only for final output, not during training)
        ensemble_pred_clipped = np.clip(ensemble_pred, 16, 85)

        return {
            'ensemble': ensemble_pred_clipped,
            'cnn_lstm': cnn_pred,
            'xgboost': xgb_pred
        }

print("CNN-LSTM model class with ensemble capability defined successfully")
print("Features: Multi-task learning, Huber loss, feature extraction, XGBoost ensemble")

## Complete Training Pipeline

The main predictor class

In [ ]:
class CODE15AgePredictorFinal:
    """
    Complete CODE-15 age predictor implementing all traveling fixes.
    This is the main interface that brings together all components.
    """

    def __init__(self, data_dir, metadata_path, output_dir='./results_code15'):
        self.dataset = CODE15DatasetFinal(data_dir, metadata_path)
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.cnn_lstm = None

    def collect_data_for_xgboost(self, df, max_samples=5000, saliency_mask=None):
        """Collect actual data arrays for XGBoost training"""
        print(f"Collecting up to {max_samples:,} samples for XGBoost...")

        X_data = []
        y_data = []
        collected = 0

        data_gen, _ = self.dataset.create_weighted_data_generator(
            df, batch_size=32, shuffle=False, is_train=False, saliency_mask=saliency_mask
        )

        for X_batch, y_batch_dict, w_batch_dict in data_gen():
            X_data.extend(X_batch)
            y_data.extend(y_batch_dict['age_output'])
            collected += len(X_batch)

            if collected >= max_samples:
                break

        X_data = np.array(X_data[:max_samples])
        y_data = np.array(y_data[:max_samples])

        print(f"✓ Collected {len(X_data):,} samples for XGBoost")
        return X_data, y_data

    def train(self, epochs=50, batch_size=32, xgb_samples=5000, use_saliency_guidance=True, 
              saliency_update_frequency=10):
        """
        Complete training pipeline with all traveling fixes and optimizations.
        """
        print("="*80)
        print("TRAINING CODE-15 AGE PREDICTION MODEL")
        print("="*80)
        print("FEATURES IMPLEMENTED:")
        print("✓ Optimal ECG preprocessing (0.5-40Hz bandpass, 400Hz→100Hz)")
        print("✓ 5-year bin equalization (15 bins: 16-20, ..., 85-90)")
        

## Comprehensive Evaluation and Results

Final evaluation pipeline with detailed metrics, saliency visualization, and Excel export.

In [1]:
def evaluate_comprehensive(self, test_df, batch_size=32):
    """Comprehensive evaluation with ensemble predictions and detailed analysis"""
    print("Running comprehensive evaluation with ensemble predictions...")

    gpu_detected = len(tf.config.experimental.list_physical_devices('GPU')) > 0
    eval_batch_size = batch_size * 4 if gpu_detected else batch_size

    all_preds_ensemble = []
    all_preds_cnn = []
    all_preds_xgb = []
    all_labels = []
    all_ecgs = []

    test_gen, n_valid_test = self.dataset.create_weighted_data_generator(
        test_df, eval_batch_size, shuffle=False, is_train=False
    )
    num_batches = n_valid_test // eval_batch_size

    device_name = '/GPU:0' if gpu_detected else '/CPU:0'
    with tf.device(device_name):
        for X_batch, y_batch_dict, w_batch_dict in tqdm(test_gen(), desc="Evaluation", total=num_batches):
            X_batch_gpu = tf.cast(X_batch, tf.float32)

            # Get ensemble predictions
            pred_dict = self.cnn_lstm.predict_ensemble(X_batch_gpu)

            all_preds_ensemble.extend(pred_dict['ensemble'])
            all_preds_cnn.extend(pred_dict['cnn_lstm'])
            all_preds_xgb.extend(pred_dict['xgboost'])
            
            # Extract labels
            y_ages = y_batch_dict['age_output'].numpy() if hasattr(y_batch_dict['age_output'], 'numpy') else y_batch_dict['age_output']
            all_labels.extend(y_ages)
            all_ecgs.extend(X_batch_gpu.numpy() if hasattr(X_batch_gpu, 'numpy') else X_batch_gpu)

            if len(all_labels) >= n_valid_test:
                break

    # Convert to numpy arrays
    results_data = {
        'ensemble': np.array(all_preds_ensemble),
        'cnn_lstm': np.array(all_preds_cnn),
        'xgboost': np.array(all_preds_xgb),
        'labels': np.array(all_labels),
        'ecgs': np.array(all_ecgs)
    }

    print(f"✓ Evaluation completed on {len(results_data['labels']):,} samples")

    # Calculate comprehensive metrics
    results = self._calculate_comprehensive_metrics(results_data)

    # Generate saliency maps for visualization
    saliency_maps = self._generate_saliency_maps(results_data['ecgs'], results_data['labels'], n_samples=10)

    # Save all results
    self._save_results_to_excel(results, saliency_maps)

    return results

def _calculate_comprehensive_metrics(self, results_data):
    """Calculate detailed metrics including per-decile analysis"""
    comprehensive_results = {}
    y_true = results_data['labels']

    for model_name, predictions in [
        ('Ensemble', results_data['ensemble']), 
        ('CNN-LSTM', results_data['cnn_lstm']), 
        ('XGBoost', results_data['xgboost'])
    ]:
        y_pred = predictions
        
        # Core regression metrics
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        pearson_r, _ = pearsonr(y_true, y_pred)
        psnr = compute_psnr(y_true, y_pred)

        # F1 score using age binning
        age_bins_true = np.array([age_to_bin_index(age) for age in y_true])
        age_bins_pred = np.array([age_to_bin_index(age) for age in y_pred])
        f1 = f1_score(age_bins_true, age_bins_pred, average='weighted')

        # Per-decile analysis for detailed performance assessment
        deciles = np.percentile(y_true, np.arange(0, 101, 10))
        decile_metrics = []

        for i in range(len(deciles)-1):
            mask = (y_true >= deciles[i]) & (y_true < deciles[i+1])
            if np.sum(mask) > 0:
                decile_mae = mean_absolute_error(y_true[mask], y_pred[mask])
                decile_rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
                decile_r2 = r2_score(y_true[mask], y_pred[mask])
                decile_psnr = compute_psnr(y_true[mask], y_pred[mask])
                decile_bias = np.mean(y_pred[mask] - y_true[mask])
                
                decile_metrics.append({
                    'decile': i+1,
                    'age_range': f"{deciles[i]:.1f}-{deciles[i+1]:.1f}",
                    'mae': decile_mae,
                    'rmse': decile_rmse,
                    'r2': decile_r2,
                    'psnr': decile_psnr,
                    'bias': decile_bias,
                    'n_samples': np.sum(mask)
                })

        comprehensive_results[model_name] = {
            'overall_mae': mae,
            'overall_rmse': rmse,
            'overall_r2': r2,
            'overall_psnr': psnr,
            'pearson_r': pearson_r,
            'f1_score': f1,
            'n_samples': len(y_true),
            'decile_metrics': decile_metrics
        }

    return comprehensive_results

def _generate_saliency_maps(self, X_sample, y_sample, n_samples=10):
    """Generate saliency maps for model interpretability"""
    print("Generating saliency maps for interpretability...")
    
    indices = np.random.choice(len(X_sample), min(n_samples, len(X_sample)), replace=False)
    saliency_results = []

    for i, idx in enumerate(indices):
        x = X_sample[idx:idx+1]
        y_true = y_sample[idx]

        x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(x_tensor)
            model_outputs = self.cnn_lstm.model(x_tensor)
            y_pred = model_outputs[0]  # Age output

        gradients = tape.gradient(y_pred, x_tensor)
        if gradients is not None:
            gradients_cpu = gradients.numpy()[0]
            x_cpu = x[0]
            y_pred_cpu = y_pred.numpy()[0][0]

            saliency = np.abs(gradients_cpu)

            saliency_results.append({
                'ecg': x_cpu,
                'saliency': saliency,
                'y_true': y_true,
                'y_pred': y_pred_cpu
            })

            # Save individual saliency plot
            self._plot_saliency_map(x_cpu, saliency, y_true, y_pred_cpu,
                                   os.path.join(self.output_dir, f'saliency_map_{i+1}.png'))

    return saliency_results

def _plot_saliency_map(self, ecg, saliency, y_true, y_pred, filename):
    """Plot saliency map overlaid on ECG signals"""
    lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

    fig, axes = plt.subplots(6, 2, figsize=(15, 12))
    axes = axes.flatten()

    for lead in range(12):
        ax = axes[lead]
        time_points = np.arange(len(ecg[:, lead])) / 100  # 100Hz sampling

        # Plot ECG signal
        ax.plot(time_points, ecg[:, lead], 'k-', linewidth=1, alpha=0.8)

        # Overlay saliency as colored dots
        max_saliency = np.max(saliency[:, lead])
        if max_saliency > 0:
            dot_sizes = (saliency[:, lead] / max_saliency) * 50
            ax.scatter(time_points, ecg[:, lead], s=dot_sizes, c='red', alpha=0.6)

        ax.set_title(f'{lead_names[lead]}', fontsize=10)
        ax.set_xlim(0, 10)
        ax.grid(True, alpha=0.3)

        if lead >= 10:
            ax.set_xlabel('Time (s)')

    plt.suptitle(f'Saliency Map - True Age: {y_true:.1f}, Predicted: {y_pred:.1f}', fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

def _save_results_to_excel(self, results, saliency_maps):
    """Save comprehensive results to Excel with fallback to CSV"""
    output_file = os.path.join(self.output_dir, 'code15_age_prediction_results_comprehensive.xlsx')

    # Prepare overall results
    overall_data = []
    for model_name, metrics in results.items():
        overall_data.append({
            'Model': model_name,
            'MAE': metrics['overall_mae'],
            'RMSE': metrics['overall_rmse'],
            'R²': metrics['overall_r2'],
            'PSNR': metrics['overall_psnr'],
            'Pearson_R': metrics['pearson_r'],
            'F1_Score': metrics['f1_score'],
            'N_Samples': metrics['n_samples']
        })

    overall_df = pd.DataFrame(overall_data)

    # Prepare saliency summary
    saliency_data = []
    for i, smap in enumerate(saliency_maps):
        saliency_data.append({
            'Sample': i+1,
            'True_Age': smap['y_true'],
            'Predicted_Age': smap['y_pred'],
            'Error': abs(smap['y_true'] - smap['y_pred']),
            'Max_Saliency': np.max(smap['saliency']),
            'Mean_Saliency': np.mean(smap['saliency'])
        })

    saliency_df = pd.DataFrame(saliency_data)

    # Try saving as Excel, fallback to CSV
    try:
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            overall_df.to_excel(writer, sheet_name='Overall_Results', index=False)
            
            for model_name, metrics in results.items():
                decile_df = pd.DataFrame(metrics['decile_metrics'])
                sheet_name = f'{model_name}_Deciles'[:31]
                decile_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            saliency_df.to_excel(writer, sheet_name='Saliency_Maps', index=False)

        print(f"✓ Results saved to: {output_file}")
        
    except Exception as e:
        print(f"Excel save failed ({e}), saving as CSV files...")
        
        overall_df.to_csv(os.path.join(self.output_dir, 'overall_results.csv'), index=False)
        saliency_df.to_csv(os.path.join(self.output_dir, 'saliency_maps.csv'), index=False)
        
        for model_name, metrics in results.items():
            decile_df = pd.DataFrame(metrics['decile_metrics'])
            filename = f'{model_name.lower()}_deciles.csv'
            decile_df.to_csv(os.path.join(self.output_dir, filename), index=False)
        
        print(f"✓ CSV files saved to: {self.output_dir}/")

# Add evaluation methods to the predictor class
CODE15AgePredictorFinal.evaluate_comprehensive = evaluate_comprehensive
CODE15AgePredictorFinal._calculate_comprehensive_metrics = _calculate_comprehensive_metrics
CODE15AgePredictorFinal._generate_saliency_maps = _generate_saliency_maps
CODE15AgePredictorFinal._plot_saliency_map = _plot_saliency_map
CODE15AgePredictorFinal._save_results_to_excel = _save_results_to_excel

print("Comprehensive evaluation pipeline implemented successfully")

# MAIN EXECUTION CODE
def main():
    """
    Main execution function - UPDATE THESE PATHS FOR YOUR ENVIRONMENT
    """
    
    # UPDATE THESE PATHS TO YOUR DATA LOCATION
    DATA_DIR = r"C:\Users\chaud\OneDrive\Desktop\CODE 15 (data and models)\Extracted data CODE 15"
    METADATA_PATH = r"C:\Users\chaud\OneDrive\Desktop\CODE 15 (data and models)\Extracted data CODE 15\exams.csv"
    OUTPUT_DIR = "./results_code15_comprehensive"

    print("CODE-15 AGE PREDICTION WITH COMPREHENSIVE EVALUATION")
    print("="*80)
    print("IMPLEMENTATION FEATURES:")
    print("✓ Optimal ECG preprocessing pipeline")
    print("✓ 5-year bin equalization for balanced training") 
    print("✓ Multi-task CNN-BiLSTM architecture")
    print("✓ Saliency-guided attention mechanisms")
    print("✓ XGBoost ensemble for improved accuracy")
    print("✓ Comprehensive evaluation and visualization")
    print("="*80)

    try:
        # Create predictor instance
        predictor = CODE15AgePredictorFinal(DATA_DIR, METADATA_PATH, OUTPUT_DIR)

        # Train model with all optimizations
        print("\nSTARTING TRAINING PIPELINE...")
        history, (train_df, val_df, test_df), standalone_results = predictor.train(
            epochs=50, 
            batch_size=24,  # Ensure good bin coverage (24 > 15 bins)
            xgb_samples=5000,
            use_saliency_guidance=True,
            saliency_update_frequency=10
        )

        # Comprehensive evaluation
        print("\nSTARTING COMPREHENSIVE EVALUATION...")
        results = predictor.evaluate_comprehensive(test_df)

        # Print final results summary
        print("\n" + "="*80)
        print("FINAL RESULTS SUMMARY")
        print("="*80)

        # Print standalone CNN-LSTM results
        print(f"\nStandalone CNN-LSTM Performance:")
        print(f"  MAE: {standalone_results['mae']:.3f} years")
        print(f"  RMSE: {standalone_results['rmse']:.3f} years") 
        print(f"  R²: {standalone_results['r2']:.3f}")
        print(f"  PSNR: {standalone_results['psnr']:.1f} dB")
        print(f"  Bin Accuracy: {standalone_results['bin_accuracy']:.3f}")

        # Print ensemble results
        for model_name, metrics in results.items():
            print(f"\n{model_name} Performance:")
            print(f"  MAE: {metrics['overall_mae']:.3f} years")
            print(f"  RMSE: {metrics['overall_rmse']:.3f} years")
            print(f"  R²: {metrics['overall_r2']:.3f}")
            print(f"  PSNR: {metrics['overall_psnr']:.1f} dB")
            print(f"  Pearson R: {metrics['pearson_r']:.3f}")
            print(f"  F1 Score: {metrics['f1_score']:.3f}")
            print(f"  Samples: {metrics['n_samples']:,}")

        print(f"\n  All results and visualizations saved to: {OUTPUT_DIR}/")
        print(f"  Generated files:")
        print(f"  - Comprehensive metrics (Excel/CSV)")
        print(f"  - Saliency maps (PNG)")
        print(f"  - Best model weights (H5)")

        print("\n" + "="*80)
        print("TRAINING AND EVALUATION COMPLETED SUCCESSFULLY!")
        print("All traveling fixes and optimizations applied")
        print("="*80)

    except Exception as e:
        print(f"\n Error during execution: {e}")
        import traceback
        traceback.print_exc()

# Uncomment the line below to run the main function
# main()

print("Complete notebook conversion ready - all chunks implemented!")
print("To run: Uncomment the main() call at the end and update the data paths")

NameError: name 'CODE15AgePredictorFinal' is not defined